In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# 재현성(Reproducibility) 확보: 난수 생성 시드를 42로 고정하여 실행할 때마다 동일한 가중치 초기화와 동일한 결과를 얻도록 합니다.
torch.manual_seed(42)

# ==========================================
# 1. Vocab(단어 사전) 및 토크나이저 / 디토크나이저 정의
# ==========================================
# 문자열 토큰을 모델이 처리할 수 있는 고유한 정수 ID로 변환하기 위한 단어 사전입니다. (총 39개 토큰)
vocab = {
    "[PAD]": 0,
    "[UNK]": 1,
    "[BOS]": 2,
    "[EOS]": 3,
    ".": 4,
    "가": 5,
    "개발자": 6,
    "구출했다": 7,
    "구매자": 8,
    "기자": 9,
    "간병했다": 10,
    "감독": 11,
    "독촉했다": 12,
    "모방했다": 13,
    "배우": 14,
    "변호사": 15,
    "비판했다": 16,
    "사원": 17,
    "선배": 18,
    "선수": 19,
    "설득했다": 20,
    "승객": 21,
    "의사": 22,
    "원고": 23,
    "운전자": 24,
    "이": 25,
    "인공지능": 26,
    "을": 27,
    "조사했다": 28,
    "취재했다": 29,
    "팀장": 30,
    "통제했다": 31,
    "평가했다": 32,
    "평론가": 33,
    "판매자": 34,
    "화가": 35,
    "환자": 36,
    "후배": 37,
    "를": 38,
}


# 텍스트 문장을 토큰 단위로 분리하고 이를 정수 ID 리스트로 변환하는 토크나이저 함수입니다.
def tokenize_and_encode(text, vocab):
    # 문장의 시작을 알리는 특수 토큰 [BOS]를 토큰 리스트의 맨 앞에 추가합니다.
    tokens = ["[BOS]"]

    # 한국어 어절 분리를 위해 기준이 되는 조사 목록을 정의합니다.
    josa_list = ["가", "이", "를", "을"]

    # 마침표 분리를 용이하게 하기 위해 마침표 앞에 공백을 둔 후, 띄어쓰기 기준으로 어절을 쪼냅니다.
    words = text.replace(".", " .").split()

    # 어절을 순회하며 체언과 조사를 분리하는 작업을 수행합니다.
    for word in words:
        separated = False
        for josa in josa_list:
            # 어절이 조사로 끝나고 어절 길이가 조사 길이보다 긴 경우 체언과 조사를 split 처리합니다.
            if word.endswith(josa) and len(word) > len(josa):
                tokens.extend([word[: -len(josa)], josa])
                separated = True
                break
        # 조사가 분리되지 않은 단어는 그대로 토큰에 추가합니다.
        if not separated:
            tokens.append(word)

    # 문장의 끝을 알리는 특수 토큰 [EOS]를 토큰 리스트 끝에 추가합니다.
    tokens.append("[EOS]")
    # 문장 토큰들을 사전(vocab)에서 찾아 정수 ID로 매핑하여 반환합니다. (없을 경우 [UNK] ID=1 사용)
    # Return: tokens(List[str]), ids(List[int])
    return tokens, [vocab.get(t, vocab["[UNK]"]) for t in tokens]


# 배치(Batch) 내 문장들의 길이를 가장 긴 문장에 맞춰 패딩([PAD]=0)으로 채워주는 함수입니다.
def pad_sequences(ids_list, pad_id=0):
    # 입력된 문장 ID 리스트들 중 가장 긴 문장의 길이를 구합니다. (예: max_len = 8)
    max_len = max(len(ids) for ids in ids_list)

    # 부족한 길이만큼 pad_id(0)를 뒤에 붙여 2차원 LongTensor 형태로 변환하여 반환합니다.
    # Return Shape: [batch_size, max_len] = [10, 8]
    return torch.tensor(
        [ids + [pad_id] * (max_len - len(ids)) for ids in ids_list], dtype=torch.long
    )


# 예측된 토큰 문자열 리스트를 한국어 문법 및 띄어쓰기에 맞게 다시 결합하는 후처리 함수입니다.
def detokenize(tokens):
    # 결합 시 띄어쓰기를 하지 않고 붙여 쓸 조사 목록을 지정합니다.
    josa_list = ["가", "이", "를", "을"]

    # 문장 재구성 시 제외시킬 특수 토큰들입니다.
    special_tokens = ["[BOS]", "[EOS]", "[PAD]", "[UNK]"]

    # 불필요한 특수 토큰들을 원본 리스트에서 제거합니다.
    clean_tokens = [t for t in tokens if t not in special_tokens]

    result = []
    for token in clean_tokens:
        # 첫 번째 단어는 그대로 추가합니다.
        if not result:
            result.append(token)
            continue

        # 토큰이 조사이거나 마침표인 경우 띄어쓰지 않고 앞 단어 뒤에 바로 붙입니다.
        if token in josa_list or token == ".":
            result[-1] += token
        else:
            # 일반 단어인 경우 띄어쓰기로 결합합니다.
            result.append(token)

    # 최종 재결합된 문자열을 반환합니다.
    return " ".join(result)


# ==========================================
# 2. Transformer Embedding 모듈
# ==========================================
# 단어 ID를 임베딩 벡터로 변환하고 위치 정보(Positional Encoding)를 더해주는 모듈입니다.
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model=128, max_len=100):
        super().__init__()

        # 39, 128

        # d_model: 모델 내부 표현 차원 (값: 128)
        self.d_model = d_model

        # 정수 ID를 d_model 차원의 학습 가능한 벡터로 룩업(Lookup)하는 임베딩 레이어
        # Weight Shape: [vocab_size, d_model] = [39, 128]
        self.word_emb = nn.Embedding(vocab_size, d_model)

        # 위치 정보를 저장할 Positional Encoding 행렬을 초기화합니다.
        # pe Shape: [max_len, d_model] = [100, 128]
        pe = torch.zeros(max_len, d_model)

        # 0부터 max_len-1까지의 위치 인덱스를 생성합니다.
        # position Shape: [max_len, 1] = [100, 1]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 삼각함수의 주기를 결정하는 분모(div_term)를 계산합니다.
        # div_term Shape: [d_model // 2] = [64]
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # 짝수 인덱스 차원에는 sin 함수 적용
        pe[:, 0::2] = torch.sin(position * div_term)

        # 홀수 인덱스 차원에는 cos 함수 적용
        pe[:, 1::2] = torch.cos(position * div_term)

        # 학습 매개변수가 아니므로 register_buffer로 상수에 등록하고 배치 차원을 확장합니다.
        # Buffer 'pe' Shape: [1, max_len, d_model] = [1, 100, 128]
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x Shape: [batch_size, seq_len] -> 예: [10, 8]
        seq_len = x.size(1)

        # 단어 ID를 임베딩 벡터로 바꾸고 sqrt(d_model)을 곱해 포지셔널 임베딩과 연산 시 스케일을 맞춥니다.
        # token_embeddings Shape: [batch_size, seq_len, d_model] = [10, 8, 128]
        token_embeddings = self.word_emb(x) * math.sqrt(self.d_model)

        # 입력된 문장의 시퀀스 길이에 맞춰 위치 임베딩을 슬라이싱합니다.
        # pos_embeddings Shape: [1, seq_len, d_model] = [1, 8, 128]
        pos_embeddings = self.pe[:, :seq_len]

        # 단어 임베딩 벡터와 위치 임베딩 벡터를 요소별 더하기(Element-wise Add)하여 반환합니다.
        # Return Shape: [batch_size, seq_len, d_model] = [10, 8, 128]
        return token_embeddings + pos_embeddings


# ==========================================
# 3. Multi-Head Self-Attention 및 Cross-Attention 모듈
# ==========================================
# 단일 입력 x만 받아 내부에서 Q, K, V로 분기하는 Self-Attention 전용 클래스입니다.
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        # d_model(128)이 헤드 수 num_heads(4)로 나누어지는지 확인
        assert d_model % num_heads == 0
        self.d_model = d_model  # 모델 차원 = 128
        self.num_heads = num_heads  # 헤드 개수 = 4
        self.d_k = d_model // num_heads  # 각 헤드별 차원 = 32

        # 단일 입력 x로부터 Q, K, V를 가각 선형 투영(Linear Projection)하기 위한 레이어
        # Weight Shape: [d_model, d_model] = [128, 128]
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # 여러 헤드의 출력을 하나로 결합한 후 투영하는 출력 선형 레이어
        # Weight Shape: [d_model, d_model] = [128, 128]
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        # x Shape: [batch_size, seq_len, d_model] -> 예: [10, 8, 128]
        batch_size = x.size(0)

        # 1) 입력 x를 선형 투영한 후, [batch, num_heads, seq_len, d_k] 형태로 차원을 분할 변경합니다.
        # Q, K, V Shape: [batch_size, num_heads, seq_len, d_k] = [10, 4, 8, 32]
        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2) Q와 K^T를 내적 연산하고 sqrt(d_k)로 나누어 Scaled Dot-Product 점수를 계산합니다.
        # scores Shape: [batch_size, num_heads, seq_len, seq_len] = [10, 4, 8, 8]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 3) 마스크 조건이 주어졌을 때 False(0)인 위치의 점수를 음수 무한대(-inf)로 치환합니다.
        if mask is not None:
            # scores Shape: [batch_size, num_heads, seq_len, seq_len]
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # 4) Softmax를 취해 각 토큰 간 어텐션 가중치 확률 분포(0~1)를 생성합니다.
        # attn_probs Shape: [batch_size, num_heads, seq_len, seq_len] = [10, 4, 8, 8]
        attn_probs = torch.softmax(scores, dim=-1)

        # 5) 가중치 확률 분포와 Value(V) 벡터를 내적하여 문맥이 반영된 출력을 계산합니다.
        # attn_output Shape: [batch_size, num_heads, seq_len, d_k] = [10, 4, 8, 32]
        attn_output = torch.matmul(attn_probs, V)

        # 6) 여러 개의 헤드로 나누어져 있던 결과를 원래 차원(d_model=128)으로 펼쳐서 결합(Concatenate)합니다.
        # attn_output Shape: [batch_size, seq_len, d_model] = [10, 8, 128]
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, -1, self.d_model)

        # 7) 최종 Output Linear 투영을 거쳐 최종 Self-Attention 표현값을 생성합니다.
        # output Shape: [batch_size, seq_len, d_model] = [10, 8, 128]
        output = self.w_o(attn_output)

        return output, attn_probs


# Q는 디코더에서, K와 V는 인코더 출력(memory)에서 오는 Cross-Attention 클래스입니다.
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Q, K, V 입력을 변환할 선형 레이어들
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        # q Shape (디코더): [batch_size, tgt_len, d_model] = [10, 7, 128]
        # k, v Shape (인코더): [batch_size, src_len, d_model] = [10, 8, 128]
        batch_size = q.size(0)

        # 1) Q(디코더 타겟), K(인코더 메모리), V(인코더 메모리)를 각각 독립적으로 투영 및 분할합니다.
        # Q Shape: [batch_size, num_heads, tgt_len, d_k] = [10, 4, 7, 32]
        # K, V Shape: [batch_size, num_heads, src_len, d_k] = [10, 4, 8, 32]
        Q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2) 디코더 Q와 인코더 K 사이의 어텐션 스코어를 내적 계산합니다.
        # scores Shape: [batch_size, num_heads, tgt_len, src_len] = [10, 4, 7, 8]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 3) 인코더 패딩 마스크가 주어졌을 경우 무효한 위치를 -inf로 채웁니다.
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # 4) Softmax 적용하여 인코더 토큰 참조 확률 계산
        # attn_probs Shape: [batch_size, num_heads, tgt_len, src_len] = [10, 4, 7, 8]
        attn_probs = torch.softmax(scores, dim=-1)

        # 5) 확률값과 인코더 Value(V) 벡터를 곱하여 가중합 산출
        # attn_output Shape: [batch_size, num_heads, tgt_len, d_k] = [10, 4, 7, 32]
        attn_output = torch.matmul(attn_probs, V)

        # 6) 헤드 재결합
        # attn_output Shape: [batch_size, tgt_len, d_model] = [10, 7, 128]
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, -1, self.d_model)

        # 7) 최종 선형 투영
        # output Shape: [batch_size, tgt_len, d_model] = [10, 7, 128]
        output = self.w_o(attn_output)

        return output, attn_probs


# ==========================================
# 4. Feed-Forward & Sub-Layers
# ==========================================
# 어텐션 층 이후 각 토큰 위치별로 독립적으로 적용되는 2층 신경망(Feed Forward Network) 모듈입니다.
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # 1차 선형 레이어: 은닉 차원을 d_model(128)에서 d_ff(512)로 확장
        self.fc1 = nn.Linear(d_model, d_ff)
        # 2차 선형 레이어: 차원을 d_ff(512)에서 원래 d_model(128)로 축소 복원
        self.fc2 = nn.Linear(d_ff, d_model)
        # 비선형 활성화 함수 ReLU
        self.relu = nn.ReLU()
        # 과적합 방지를 위한 Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x Shape: [batch_size, seq_len, d_model] = [10, 8, 128]
        # fc1(x) -> [10, 8, 512] -> relu -> dropout -> fc2 -> [10, 8, 128]
        # Return Shape: [batch_size, seq_len, d_model] = [10, 8, 128]
        return self.fc2(self.dropout(self.relu(self.fc1(x))))


# ==========================================
# 5. Transformer Encoder & Decoder Layer
# ==========================================
# 인코더의 단일 레이어 블록입니다. (Self-Attention + FFN + Residual Connection)
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # 단일 입력 전용 Self-Attention 모듈
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads)
        # Position-wise Feed Forward Network
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)

        # 정규화 및 드롭아웃 레이어
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x Shape: [batch_size, enc_seq_len, d_model] = [10, 8, 128]

        # 1) Self-Attention 수행 (단일 입력 x만 전달)
        # attn_out Shape: [10, 8, 128], attn_weights Shape: [10, 4, 8, 8]
        attn_out, attn_weights = self.self_attn(x, mask=mask)

        # 잔차 연결(Residual Connection) 및 Layer Normalization 수행
        # x Shape: [batch_size, enc_seq_len, d_model] = [10, 8, 128]
        x = self.norm1(x + self.dropout1(attn_out))

        # 2) Feed-Forward Network 연산 수행
        # ffn_out Shape: [batch_size, enc_seq_len, d_model] = [10, 8, 128]
        ffn_out = self.ffn(x)

        # 두 번째 잔차 연결 및 Layer Normalization 수행
        # x Shape: [batch_size, enc_seq_len, d_model] = [10, 8, 128]
        x = self.norm2(x + self.dropout2(ffn_out))

        return x, attn_weights


# 디코더의 단일 레이어 블록입니다. (Masked Self-Attention + Cross-Attention + FFN)
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # 1) Masked Self-Attention (디코더 자기 자신 연산용 - 단일 입력)
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads)

        # 2) Cross-Attention (인코더 출력 memory 연산용 - 3개 입력)
        self.cross_attn = MultiHeadCrossAttention(d_model, num_heads)

        # 3) Feed Forward
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        # tgt Shape (디코더 입력): [batch_size, dec_seq_len, d_model] = [10, 7, 128]
        # memory Shape (인코더 최종 출력): [batch_size, enc_seq_len, d_model] = [10, 8, 128]

        # Sub-layer 1: Masked Self-Attention 수행 (단일 입력 tgt 전달)
        # self_attn_out Shape: [10, 7, 128], self_attn_weights Shape: [10, 4, 7, 7]
        self_attn_out, self_attn_weights = self.self_attn(tgt, mask=tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(self_attn_out))

        # Sub-layer 2: Cross-Attention 수행 (Q=tgt, K=memory, V=memory)
        # cross_attn_out Shape: [10, 7, 128], cross_attn_weights Shape: [10, 4, 7, 8]
        cross_attn_out, cross_attn_weights = self.cross_attn(
            tgt, memory, memory, mask=memory_mask
        )
        tgt = self.norm2(tgt + self.dropout2(cross_attn_out))

        # Sub-layer 3: Feed-Forward 연산 및 잔차 연결
        # ffn_out Shape: [10, 7, 128]
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout3(ffn_out))

        # tgt Shape: [batch_size, dec_seq_len, d_model] = [10, 7, 128]
        return tgt, self_attn_weights, cross_attn_weights


# ==========================================
# 6. 전체 Transformer Model (Encoder + Decoder + Generator)
# ==========================================
class TransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, d_ff=512, dropout=0.1):
        super().__init__()

        # 임베딩 계층 생성
        self.embedding = TransformerEmbedding(vocab_size, d_model)
        # 인코더 레이어 생성
        self.encoder = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        # 디코더 레이어 생성
        self.decoder = TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
        # 디코더의 히든 표현(128차원)을 Vocab 크기(39개)로 투영하는 최종 선형 분류기
        self.generator = nn.Linear(d_model, vocab_size)

    # 미래 단어 조회를 방지하기 위한 Causal Mask(하삼각 행렬) 생성 메서드
    def generate_causal_mask(self, seq_len, device):
        # 대각선 포함 아래쪽 삼각 요소를 1(True)로 채운 마스크 행렬 생성
        # mask Shape: [seq_len, seq_len] = [7, 7]
        mask = torch.tril(torch.ones((seq_len, seq_len), device=device)).bool()

        # 배치 및 헤드 차원 추가하여 확장
        # Return Shape: [1, 1, seq_len, seq_len] = [1, 1, 7, 7]
        return mask.unsqueeze(0).unsqueeze(1)

    def forward(self, src_ids, tgt_ids, pad_id=0):
        # src_ids Shape: [batch_size, enc_seq_len] = [10, 8]
        # tgt_ids Shape: [batch_size, dec_seq_len] = [10, 7]

        # 1) 인코더 입력 패딩 마스크 생성 (패딩 위치[PAD=0]는 False로 처리)
        # src_mask Shape: [batch_size, 1, 1, enc_seq_len] = [10, 1, 1, 8]
        src_mask = (src_ids != pad_id).unsqueeze(1).unsqueeze(2)
        # 2) 디코더 패딩 마스크 생성
        # tgt_pad_mask Shape: [batch_size, 1, 1, dec_seq_len] = [10, 1, 1, 7]
        tgt_pad_mask = (tgt_ids != pad_id).unsqueeze(1).unsqueeze(2)
        # 3) 디코더 Causal 마스크 생성
        # causal_mask Shape: [1, 1, dec_seq_len, dec_seq_len] = [1, 1, 7, 7]
        causal_mask = self.generate_causal_mask(tgt_ids.size(1), tgt_ids.device)
        # 4) 디코더 최종 마스크 = 패딩 마스크 AND Causal 마스크 결합
        # tgt_mask Shape: [batch_size, 1, dec_seq_len, dec_seq_len] = [10, 1, 7, 7]
        tgt_mask = tgt_pad_mask & causal_mask
        # 5) 인코더 순전파 수행 (Self-Attention 연산 실행)
        # src_emb Shape: [10, 8, 128]
        src_emb = self.embedding(src_ids)
        # memory Shape: [10, 8, 128]
        memory, enc_attn = self.encoder(src_emb, mask=src_mask)

        # 6) 디코더 순전파 수행 (Masked Self-Attention 및 Cross-Attention 실행)
        # tgt_emb Shape: [10, 7, 128]
        tgt_emb = self.embedding(tgt_ids)
        # dec_output Shape: [10, 7, 128]
        dec_output, dec_self_attn, cross_attn = self.decoder(
            tgt_emb, memory, tgt_mask=tgt_mask, memory_mask=src_mask
        )

        # 7) Generator를 통해 각 위치별 Vocab 단어 예측 Logits 산출
        # logits Shape: [batch_size, dec_seq_len, vocab_size] = [10, 7, 39]
        logits = self.generator(dec_output)

        return logits, enc_attn, dec_self_attn, cross_attn


# ==========================================
# 7. 학습(Training) 및 결과 확인
# ==========================================
if __name__ == "__main__":
    # 학습에 사용될 10개의 데이터셋 (입력 문장, 타겟 문장)
    sentences = [
        ("팀장이 사원을 평가했다.", "사원이 팀장을 평가했다."),
        ("선수가 감독을 설득했다.", "감독이 선수를 설득했다."),
        ("변호사가 원고를 조사했다.", "원고가 변호사를 조사했다."),
        ("기자가 배우를 취재했다.", "배우가 기자를 취재했다."),
        ("구매자가 판매자를 독촉했다.", "판매자가 구매자를 독촉했다."),
        ("의사가 환자를 간병했다.", "환자가 의사를 간병했다."),
        ("화가가 평론가를 비판했다.", "평론가가 화가를 비판했다."),
        ("운전자가 승객을 구출했다.", "승객이 운전자를 구출했다."),
        ("선배가 후배를 모방했다.", "후배가 선배를 모방했다."),
        ("개발자가 인공지능을 통제했다.", "인공지능이 개발자를 통제했다."),
    ]

    # --- 하이퍼파라미터 설정 변수 ---
    d_model = 128  # 모델 히든 차원 수 (값: 128)
    num_heads = 4  # 멀티 헤드 개수 (값: 4)
    d_ff = 512  # Feed Forward 내부 차원 (값: 512)
    dropout = 0.1  # 드롭아웃 확률 (값: 0.1)
    epochs = 100  # 학습 반복 횟수 (값: 100)
    learning_rate = 0.001  # 학습률 (값: 0.001)
    pad_id = vocab["[PAD]"]  # 패딩 토큰 ID (값: 0)

    # 1) 데이터 전처리 (토큰화 및 정수 ID 변환)
    src_ids_list, tgt_ids_list = [], []
    for src, tgt in sentences:
        _, src_enc = tokenize_and_encode(src, vocab)
        _, tgt_enc = tokenize_and_encode(tgt, vocab)
        src_ids_list.append(src_enc)
        tgt_ids_list.append(tgt_enc)

    # 배치 입력을 위해 패딩 작업 진행
    # src_ids Shape: [batch_size=10, enc_seq_len=8]
    src_ids = pad_sequences(src_ids_list, pad_id=pad_id)
    # tgt_ids Shape: [batch_size=10, dec_seq_len=8]
    tgt_ids = pad_sequences(tgt_ids_list, pad_id=pad_id)

    # 2) Transformer 모델 객체 생성
    model = TransformerSeq2Seq(
        vocab_size=len(vocab),
        d_model=d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        dropout=dropout,
    )

    # [PAD] 토큰(0)은 손실 오차 계산에서 제외하는 CrossEntropyLoss 설정
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    # Adam 최적화 가중치 옵티마이저 정의
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # 3) 모델 학습 진행
    model.train()
    print("=" * 65)
    print("Transformer Training Process Start")
    print("=" * 65)

    for epoch in range(1, epochs + 1):
        # 각 에포크 시작 시 이전 경사도(Gradient)를 초기화
        optimizer.zero_grad()

        # Teacher Forcing 기법 적용
        # dec_input Shape: [batch_size=10, dec_seq_len-1=7] -> [BOS] 포함, [EOS] 제외
        dec_input = tgt_ids[:, :-1]

        # targets Shape: [batch_size=10, dec_seq_len-1=7] -> [BOS] 제외, [EOS] 포함
        targets = tgt_ids[:, 1:]

        # 모델 forward 예측 수행
        # logits Shape: [batch_size=10, dec_seq_len=7, vocab_size=39]
        logits, _, _, _ = model(src_ids, dec_input, pad_id=pad_id)

        # CrossEntropyLoss 계산을 위해 2D 평탄화 진행
        # logits reshaped Shape: [70, 39], targets reshaped Shape: [70]
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        # 역전파(Backpropagation) 수행하여 기울기 산출
        loss.backward()

        # 옵티마이저 가중치 파라미터 업데이트
        optimizer.step()

        # 손실값 출력 (20 에포크 주기로 확인)
        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch [{epoch:3d}/{epochs}] ---> Loss: {loss.item():.4f}")

    # ==========================================
    # 8. 학습 완료 후 출력 결과 추론 테스트
    # ==========================================
    # 평가 모드로 전환 (Dropout 등 비활성화)
    model.eval()

    # ID를 다시 단어 토큰 문자열로 바꾸는 역방향 단어 사전 정의
    id_to_vocab = {v: k for k, v in vocab.items()}

    print("\n" + "=" * 65)
    print("Trained Transformer Inference Check (Target vs Prediction)")
    print("=" * 65)

    # 평가 시 Gradient 추적을 중단하여 메모리와 연산 속도 향상
    with torch.no_grad():
        # dec_input Shape: [batch_size=10, dec_seq_len=7]
        dec_input = tgt_ids[:, :-1]

        # logits Shape: [batch_size=10, dec_seq_len=7, vocab_size=39]
        logits, _, _, _ = model(src_ids, dec_input, pad_id=pad_id)

        # 가장 높은 확률을 나타낸 단어 ID 인덱스를 선택
        # predicted_ids Shape: [batch_size=10, dec_seq_len=7]
        predicted_ids = torch.argmax(logits, dim=-1)

        # 상위 3개 샘플 문장에 대한 결과 비교 출력
        for i in range(3):
            src_str = sentences[i][0]
            tgt_str = sentences[i][1]

            # 예측된 ID 리스트를 단어 문자열 목록으로 복원
            pred_tokens = ["[BOS]"] + [
                id_to_vocab[idx.item()] for idx in predicted_ids[i]
            ]

            # detokenize 처리로 자연스러운 문장 결합
            formatted_output = detokenize(pred_tokens)

            print(f"Sample {i + 1} Input  : {src_str}")
            print(f"Sample {i + 1} Target : {tgt_str}")
            print(f"Sample {i + 1} Output : {formatted_output}")
            print("-" * 65)

In [1]:
import torch
import torch.nn as nn
from torchviz import make_dot


# 예시 모델
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=10, hidden_size=20, batch_first=True)
        self.fc = nn.Linear(20, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


model = SimpleRNN()
x = torch.randn(2, 5, 10)  # [batch, seq_len, input_size]

# Forward 연산 수행
y = model(x)

# 계산 그래프 생성 및 이미지 저장
dot = make_dot(y, params=dict(model.named_parameters()))
dot.format = "png"
dot.render("model_computation_graph")  # 'model_computation_graph.png' 파일로 저장됨

'model_computation_graph.png'

In [ ]:
import torch
import torch.nn as nn
from torchinfo import summary


class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(100, 32)
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        x = self.embed(x)
        return self.fc(x.mean(dim=1))


model = SimpleClassifier()

# input_size를 지정하여 모델 구조 및 파라미터 요약 출력
summary(model, input_size=(64, 10), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
SimpleClassifier                         [64, 2]                   --
├─Embedding: 1-1                         [64, 10, 32]              3,200
├─Linear: 1-2                            [64, 2]                   66
Total params: 3,266
Trainable params: 3,266
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.21
Input size (MB): 0.01
Forward/backward pass size (MB): 0.16
Params size (MB): 0.01
Estimated Total Size (MB): 0.18

In [4]:
import numpy as np

# 데이터 개수(N)는 1개, 입력 특성(Feature) 개수는 2개로 가정합니다.
# 재현성을 위해 랜덤 시드를 고정합니다.
rng = np.random.default_rng(42)

# 가상의 입력 데이터 X (1행 2열)를 생성합니다.
X = rng.uniform(low=-1.0, high=1.0, size=(1, 2))

# 실제 정답 레이블 y를 설정합니다. (1 또는 0 어떤 값이 와도 동작합니다)
y = np.array([1])

# 가중치 W (2행 1열)를 무작위로 초기화합니다.
W = rng.uniform(low=-0.5, high=0.5, size=(2, 1))

# 편향 b (1행 1열)를 0으로 초기화합니다.
b = np.zeros((1, 1))

print("=== [준비 단계] ===")
print(f"입력 데이터 X : {X}")
print(f"가중치 W: {W}")
print(f"정답 레이블 y : {y}\n")

# ==========================================
# [1단계] 순전파 (Forward Propagation)
# ==========================================
print("=== [1단계] 순전파 (Forward) ===")

# 선형 결합을 통해 로짓(Logit) z를 계산합니다. (z = XW + b)
z = np.dot(X, W) + b
print(f"1. 로짓 (z)       : {z[0][0]:.4f}")

# 로짓에 시그모이드 함수를 적용해 확률(p)을 구합니다. (p = 1 / (1 + e^-z))
p = 1 / (1 + np.exp(-z))
print(f"2. 확률 (p)      : {p[0][0]:.4f}")

# =========================================================
#  우도(Likelihood) 수식 계산 (L = p^y * (1-p)^(1-y))
# y=1일 때는 p, y=0일 때는 1-p가 도출됩니다.
# =========================================================
likelihood = (p**y) * ((1 - p) ** (1 - y))
print(f"3. 우도 (Likelihood) : {likelihood[0][0]:.4f}")


# 오즈(Odds)를 계산합니다. (Odds = p / (1 - p))
odds = p / (1 - p)
print(f"4. 오즈 (Odds)   : {odds[0][0]:.4f}")

# 이진 교차 엔트로피(BCE) 전체 공식을 적용하여 손실(Loss)을 계산합니다.
loss = -np.log(likelihood)
print(f"5. 손실 (Loss)   : {loss[0][0]:.4f}\n")

# 손실(Loss)은 음의 로그 우도(-ln(Likelihood))와 완전히 동일합니다.
# loss = - (y * np.log(p) + (1 - y) * np.log(1 - p))

# ==========================================
# [2단계] 역전파 (Backward Propagation)
# ==========================================
print("=== [2단계] 역전파 (Backward) ===")

# 정답 변수를 반영하여 손실 함수(L)를 우도(p)로 미분합니다. (dL/dp)
dL_dp = -(y / p - (1 - y) / (1 - p))
print(f"1. 우도에 대한 손실의 기울기 (dL/dp) : {dL_dp[0][0]:.4f}")

# 시그모이드 함수(p)를 로짓(z)으로 미분합니다. (dp/dz = p * (1 - p))
dp_dz = p * (1 - p)
print(f"2. 로짓에 대한 우도의 기울기 (dp/dz) : {dp_dz[0][0]:.4f}")

# 체인 룰을 이용해 최종 로짓 오차(dL/dz)를 계산합니다. (dL/dz = dL/dp * dp_dz)
dz_chained = dL_dp * dp_dz
print(f"3. 체인 룰로 결합한 로짓 기울기      : {dz_chained[0][0]:.4f}")

# 단축 공식 결과(p - y)와 체인 룰 결합 결과가 같은지 검증합니다.
dz = p - y
print(f"4. 단축 공식 결과 (p - y)           : {dz[0][0]:.4f}\n")

# 로짓 z를 가중치 W로 미분한 값(dz/dW = X)을 이용해 가중치 기울기를 구합니다.
dW = np.dot(X.T, dz)
print(f"5. 가중치 기울기 (dW)  :\n{dW}")

# 로짓 z를 편향 b로 미분한 값(dz/db = 1)을 이용해 편향 기울기를 구합니다.
db = np.sum(dz, axis=0, keepdims=True)
print(f"6. 편향 기울기 (db)    : {db[0][0]:.4f}\n")

# ==========================================
# [3단계] 경사하강법 가중치 업데이트
# ==========================================
print("=== [3단계] 가중치 업데이트 ===")

# 학습률(Learning Rate)을 설정합니다.
lr = 0.1

# 기울기의 반대 방향으로 가중치 W를 업데이트합니다. (W = W - lr * dW)
W = W - lr * dW

# 기울기의 반대 방향으로 편향 b를 업데이트합니다. (b = b - lr * db)
b = b - lr * db

print(f"업데이트된 가중치 W :\n{W}")
print(f"업데이트된 편향 b   : {b[0][0]:.4f}")

=== [준비 단계] ===
입력 데이터 X : [[ 0.5479121  -0.12224312]]
가중치 W: [[0.35859792]
 [0.19736803]]
정답 레이블 y : [1]

=== [1단계] 순전파 (Forward) ===
1. 로짓 (z)       : 0.1724
2. 확률 (p)      : 0.5430
3. 우도 (Likelihood) : 0.5430
4. 오즈 (Odds)   : 1.1881
5. 손실 (Loss)   : 0.6107

=== [2단계] 역전파 (Backward) ===
1. 우도에 대한 손실의 기울기 (dL/dp) : -1.8417
2. 로짓에 대한 우도의 기울기 (dp/dz) : 0.2482
3. 체인 룰로 결합한 로짓 기울기      : -0.4570
4. 단축 공식 결과 (p - y)           : -0.4570

5. 가중치 기울기 (dW)  :
[[-0.25040571]
 [ 0.05586731]]
6. 편향 기울기 (db)    : -0.4570

=== [3단계] 가중치 업데이트 ===
업데이트된 가중치 W :
[[0.38363849]
 [0.1917813 ]]
업데이트된 편향 b   : 0.0457


In [5]:
import numpy as np

# 예측 확률 p와 실제 정답 y가 여러 개 있다고 가정합니다.
p = np.array([[0.8], [0.1], [0.3]])  # 모델의 예측 확률 (N=3)
y = np.array([[1], [0], [1]])  # 실제 정답 레이블

# 수치적 안정성을 위해 아주 작은 값(epsilon)을 더해 로그 안에 0이 들어가는 것을 방지합니다.
eps = 1e-15
p = np.clip(p, eps, 1 - eps)

# 이진 교차 엔트로피 공식 적용 (각 데이터별 손실)
loss_per_sample = -(y * np.log(p) + (1 - y) * np.log(1 - p))

# 전체 데이터의 평균 손실 계산
mean_loss = np.mean(loss_per_sample)

print(f"개별 데이터 손실:\n{loss_per_sample}")
print(f"최종 평균 손실:{mean_loss:.4f}")

개별 데이터 손실:
[[0.22314355]
 [0.10536052]
 [1.2039728 ]]
최종 평균 손실:0.5108


In [ ]:
import torch

# 재현성을 위해 PyTorch 시드를 고정합니다.
torch.manual_seed(42)

# 가상의 입력 데이터 X (1행 2열)를 생성합니다. (기존 uniform 범위 [-1, 1] 맞춤)
X = torch.rand((1, 2)) * 2 - 1

# 실제 정답 레이블 y를 설정합니다. (float32 타입 지정)
y = torch.tensor([[1.0]])

# 가중치 W (2행 1열)와 편향 b (1행 1열)를 무작위로 초기화하고 경사 추적(requires_grad)을 활성화합니다.
W = (torch.rand((2, 1)) - 0.5).detach().requires_grad_(True)
b = torch.zeros((1, 1), requires_grad=True)

print("=== [준비 단계] ===")
print(f"입력 데이터 X :\n{X}")
print(f"가중치 W     :\n{W.data}")
print(f"정답 레이블 y : {y.data}\n")

# ==========================================
# [1단계] 순전파 (Forward Propagation)
# ==========================================
print("=== [1단계] 순전파 (Forward) ===")

# 선형 결합을 통해 로짓(Logit) z를 계산합니다. (z = XW + b)
z = torch.matmul(X, W) + b
print(f"1. 로짓 (z)       : {z.item():.4f}")

# 로짓에 시그모이드 함수를 적용해 확률(p)을 구합니다. (p = 1 / (1 + e^-z))
p = torch.sigmoid(z)
print(f"2. 확률 (p)      : {p.item():.4f}")

# 우도(Likelihood) 수식 계산 (L = p^y * (1-p)^(1-y))
likelihood = (p**y) * ((1 - p) ** (1 - y))
print(f"3. 우도 (Likelihood) : {likelihood.item():.4f}")

# 오즈(Odds)를 계산합니다. (Odds = p / (1 - p))
odds = p / (1 - p)
print(f"4. 오즈 (Odds)   : {odds.item():.4f}")

# 이진 교차 엔트로피(BCE) 손실을 계산합니다.
# 손실(Loss)을 계산합니다. (Loss = -ln(Likelihood))
loss = -torch.log(likelihood)
print(f"5. 손실 (Loss)   : {loss.item():.4f}\n")


# PyTorch 내장 loss인 torch.nn.functional.binary_cross_entropy(p, y)와 수식 연산 결과가 일치합니다.
# loss = - (y * torch.log(p) + (1 - y) * torch.log(1 - p))
# print(f"5. 손실 (Loss)   : {loss.item():.4f}\n")

# ==========================================
# [2단계] 역전파 (Backward Propagation)
# ==========================================
print("=== [2단계] 역전파 (Backward) ===")

# ------------------------------------------
# A. 수동 미분 (기존 수학 공식으로 검증)
# ------------------------------------------
dL_dp = -(y / p - (1 - y) / (1 - p))
dp_dz = p * (1 - p)
dz_chained = dL_dp * dp_dz
dz = p - y

dW_manual = torch.matmul(X.T, dz)
db_manual = torch.sum(dz, dim=0, keepdim=True)

print(f"1. 우도에 대한 손실의 기울기 (dL/dp) : {dL_dp.item():.4f}")
print(f"2. 로짓에 대한 우도의 기울기 (dp/dz) : {dp_dz.item():.4f}")
print(f"3. 체인 룰로 결합한 로짓 기울기      : {dz_chained.item():.4f}")
print(f"4. 단축 공식 결과 (p - y)           : {dz.item():.4f}")
print(f"5. 수동 계산 가중치 기울기 (dW)   :\n{dW_manual.data}")
print(f"6. 수동 계산 편향 기울기 (db)     : {db_manual.item():.4f}\n")

# ------------------------------------------
# B. PyTorch Autograd 자동 미분 수행
# ------------------------------------------
loss.backward()

print("--- [PyTorch Autograd 검증] ---")
print(f"7. Autograd가 계산한 가중치 기울기 (W.grad) :\n{W.grad}")
print(f"8. Autograd가 계산한 편향 기울기 (b.grad)   : {b.grad.item():.4f}\n")

# ==========================================
# [3단계] 경사하강법 가중치 업데이트
# ==========================================
print("=== [3단계] 가중치 업데이트 ===")

lr = 0.1

# 파라미터 업데이트 시 연산 그래프 추적을 비활성화합니다.
with torch.no_grad():
    W -= lr * W.grad
    b -= lr * b.grad

    # 기울기 초기화
    W.grad.zero_()
    b.grad.zero_()

print(f"업데이트된 가중치 W :\n{W.data}")
print(f"업데이트된 편향 b   : {b.item():.4f}")

=== [준비 단계] ===
입력 데이터 X :
tensor([[0.7645, 0.8300]])
가중치 W     :
tensor([[-0.1171],
        [ 0.4593]])
정답 레이블 y : tensor([[1.]])

=== [1단계] 순전파 (Forward) ===
1. 로짓 (z)       : 0.2917
2. 확률 (p)      : 0.5724
3. 우도 (Likelihood) : 0.5724
4. 오즈 (Odds)   : 1.3387
5. 손실 (Loss)   : 0.5579

=== [2단계] 역전파 (Backward) ===
1. 우도에 대한 손실의 기울기 (dL/dp) : -1.7470
2. 로짓에 대한 우도의 기울기 (dp/dz) : 0.2448
3. 체인 룰로 결합한 로짓 기울기      : -0.4276
4. 단축 공식 결과 (p - y)           : -0.4276
5. 수동 계산 가중치 기울기 (dW)   :
tensor([[-0.3269],
        [-0.3549]])
6. 수동 계산 편향 기울기 (db)     : -0.4276

--- [PyTorch Autograd 검증] ---
7. Autograd가 계산한 가중치 기울기 (W.grad) :
tensor([[-0.3269],
        [-0.3549]])
8. Autograd가 계산한 편향 기울기 (b.grad)   : -0.4276

=== [3단계] 가중치 업데이트 ===
업데이트된 가중치 W :
tensor([[-0.0844],
        [ 0.4948]])
업데이트된 편향 b   : 0.0428


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 재현성을 위해 시드를 고정합니다.
torch.manual_seed(42)


# ==========================================
# [모델 정의] torch.nn.Module 상속
# ==========================================
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        # 선형 레이어 (z = XW + b 연산을 내장함)
        self.linear = nn.Linear(in_features=input_dim, out_features=1)
        # 시그모이드 활성화 함수
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """순전파 연산을 정의합니다."""
        z = self.linear(x)  # 1. 로짓 (z) 계산
        p = self.sigmoid(z)  # 2. 확률 (p) 계산
        return z, p


# ==========================================
# [준비 단계] 데이터 및 객체 생성
# ==========================================
print("=== [준비 단계] ===")

# 입력 데이터 X (1행 2열) 및 정답 y (1행 1열)
X = torch.rand((1, 2)) * 2 - 1
y = torch.tensor([[1.0]])

# 모델 인스턴스화 (입력 특성 개수 = 2)
model = BinaryClassifier(input_dim=2)

# 이전 코드와 동일한 가중치 조건 조정을 위해 초기값을 직접 지정합니다.
with torch.no_grad():
    model.linear.weight.copy_((torch.rand((1, 2)) - 0.5))
    model.linear.bias.zero_()

# 손실 함수(Loss) 및 옵티마이저(Optimizer) 설정
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.SGD(
    model.parameters(), lr=0.1
)  # 확률적 경사하강법 (Learning Rate = 0.1)

print(f"입력 데이터 X :\n{X}")
print(f"초기 가중치 W :\n{model.linear.weight.data.T}")
print(f"초기 편향 b   : {model.linear.bias.item():.4f}")
print(f"정답 레이블 y : {y.data}\n")


# ==========================================
# [1단계] 순전파 (Forward)
# ==========================================
print("=== [1단계] 순전파 (Forward) ===")

# 모델의 forward() 호출
z, p = model(X)

# 수식 기반 우도(Likelihood) 및 오즈(Odds) 계산
likelihood = (p**y) * ((1 - p) ** (1 - y))
odds = p / (1 - p)

# PyTorch 내장 BCELoss를 이용한 손실 계산
loss = criterion(p, y)

print(f"1. 로짓 (z)          : {z.item():.4f}")
print(f"2. 확률 (p)         : {p.item():.4f}")
print(f"3. 우도 (Likelihood) : {likelihood.item():.4f}")
print(f"4. 오즈 (Odds)      : {odds.item():.4f}")
print(f"5. 손실 (Loss)      : {loss.item():.4f}\n")


# ==========================================
# [2단계] 역전파 (Backward)
# ==========================================
print("=== [2단계] 역전파 (Backward) ===")

# 기존 남아있는 기울기 초기화
optimizer.zero_grad()

# Autograd 연산 그래프 역전파 수행
loss.backward()

# nn.Linear의 가중치는 (out_features, in_features) 형태이므로 전치하여 출력
print(f"가중치 기울기 (dW) :\n{model.linear.weight.grad.T}")
print(f"편향 기울기 (db)   : {model.linear.bias.grad.item():.4f}\n")


# ==========================================
# [3단계] 경사하강법 가중치 업데이트
# ==========================================
print("=== [3단계] 가중치 업데이트 ===")

# 옵티마이저를 통해 파라미터 업데이트 (W = W - lr * dW)
optimizer.step()

print(f"업데이트된 가중치 W :\n{model.linear.weight.data.T}")
print(f"업데이트된 편향 b   : {model.linear.bias.item():.4f}")

=== [준비 단계] ===
입력 데이터 X :
tensor([[0.7645, 0.8300]])
초기 가중치 W :
tensor([[ 0.1009],
        [-0.2434]])
초기 편향 b   : 0.0000
정답 레이블 y : tensor([[1.]])

=== [1단계] 순전파 (Forward) ===
1. 로짓 (z)          : -0.1249
2. 확률 (p)         : 0.4688
3. 우도 (Likelihood) : 0.4688
4. 오즈 (Odds)      : 0.8826
5. 손실 (Loss)      : 0.7576

=== [2단계] 역전파 (Backward) ===
가중치 기울기 (dW) :
tensor([[-0.4061],
        [-0.4409]])
편향 기울기 (db)   : -0.5312

=== [3단계] 가중치 업데이트 ===
업데이트된 가중치 W :
tensor([[ 0.1415],
        [-0.1993]])
업데이트된 편향 b   : 0.0531
